# TP2 - Standardized PCA

## Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

data = {
    'Mathematics': [12, 15, 8, 17, 10, 14],
    'Statistics': [11, 14, 9, 16, 12, 15],
    'Programming': [14, 16, 10, 18, 11, 13],
    'Algorithms': [13, 15, 9, 17, 10, 14],
}
students = ['S1', 'S2', 'S3', 'S4', 'S5', 'S6']
df = pd.DataFrame(data, index=students)
display(df)
print('Dimensions:', df.shape)
display(df.describe().round(3))

## Part A - PCA from scratch

In [ ]:
X = df.to_numpy(dtype=float)
Z_manual = (X - X.mean(axis=0)) / X.std(axis=0, ddof=0)
Z_manual_df = pd.DataFrame(Z_manual, index=students, columns=df.columns)
print('Standardized data:')
display(Z_manual_df.round(6))

correlation = np.corrcoef(Z_manual, rowvar=False)
print('Correlation matrix:')
display(pd.DataFrame(correlation, index=df.columns, columns=df.columns).round(6))

manual_eigenvalues, manual_eigenvectors = np.linalg.eigh(correlation)
order = np.argsort(manual_eigenvalues)[::-1]
manual_eigenvalues = manual_eigenvalues[order]
manual_eigenvectors = manual_eigenvectors[:, order]
manual_coordinates = Z_manual @ manual_eigenvectors[:, :2]
manual_coordinates_df = pd.DataFrame(manual_coordinates, index=students, columns=['PC1', 'PC2'])
manual_ratios = manual_eigenvalues / manual_eigenvalues.sum()
manual_cumulative = np.cumsum(manual_ratios)

print('Manual eigenvalues:')
display(pd.Series(manual_eigenvalues, index=['PC1', 'PC2', 'PC3', 'PC4']).round(6))
print('Manual coordinates:')
display(manual_coordinates_df.round(6))
print('Manual explained variance ratios:', manual_ratios.round(6))
print('Manual cumulative explained variance:', manual_cumulative.round(6))

## Part B - Scikit-learn PCA

In [ ]:
scaler = StandardScaler()
Z_sklearn = scaler.fit_transform(df)
pca = PCA(n_components=2)
sklearn_coordinates = pca.fit_transform(Z_sklearn)
sklearn_coordinates_df = pd.DataFrame(sklearn_coordinates, index=students, columns=['PC1', 'PC2'])
sklearn_ratios = pca.explained_variance_ratio_
sklearn_cumulative = np.cumsum(sklearn_ratios)

print('Scikit-learn coordinates:')
display(sklearn_coordinates_df.round(6))
print('Scikit-learn eigenvalues:', pca.explained_variance_.round(6))
print('Scikit-learn explained variance ratios:', sklearn_ratios.round(6))
print('Scikit-learn cumulative explained variance:', sklearn_cumulative.round(6))

## Part C - Comparison

In [ ]:
print('Manual coordinates (raw):')
display(manual_coordinates_df.round(6))
print('Scikit-learn coordinates (raw):')
display(sklearn_coordinates_df.round(6))
print('Signed difference: manual - Scikit-learn')
display((manual_coordinates_df - sklearn_coordinates_df).round(6))

print('Manual explained variance ratios:', manual_ratios.round(6))
print('Scikit-learn explained variance ratios:', sklearn_ratios.round(6))
print('Difference in the first two ratios:', np.abs(manual_ratios[:2] - sklearn_ratios).round(10))

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(manual_coordinates[:, 0], manual_coordinates[:, 1], marker='o', label='Manual PCA')
ax.scatter(sklearn_coordinates[:, 0], sklearn_coordinates[:, 1], marker='x', label='Scikit-learn PCA')
for i, student in enumerate(students):
    ax.annotate(student, (manual_coordinates[i, 0], manual_coordinates[i, 1]))
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('Raw student coordinates: Manual PCA vs Scikit-learn PCA')
ax.axhline(0, color='gray', linewidth=0.8)
ax.axvline(0, color='gray', linewidth=0.8)
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig('pca_comparison_notebook.png', dpi=150)
plt.show()

## Results

In [ ]:
variance_retained = sklearn_ratios.sum()
print(f'Variance retained by PC1 and PC2: {variance_retained * 100:.4f}%')
print(f'Variance not retained: {(1 - variance_retained) * 100:.4f}%')